# 06 — Đánh giá thống nhất (within + cross-dataset)

Nhóm 05 — Fall Detection. Bật **GPU**. Thêm dataset URFD & Le2i qua **Add Input**.


In [ ]:
# === Thiết lập chung (Kaggle) ===
import sys, os
# Cách 1: clone repo (khuyến nghị để đồng bộ code):
# !git clone https://github.com/ducqwang/FallDetection.git /kaggle/working/FD
# REPO='/kaggle/working/FD'
# Cách 2: notebook nằm sẵn trong repo:
REPO = '..' if os.path.exists('../src') else '/kaggle/working/FD'
sys.path.insert(0, REPO)
from src.utils import set_seed, load_config, on_kaggle
cfg = load_config(os.path.join(REPO, 'configs/default.yaml'))
set_seed(cfg['seed'])
# TRÊN KAGGLE: sửa cfg['paths'] trỏ tới /kaggle/input/<dataset-slug>/...
# cfg['paths']['urfd_root'] = '/kaggle/input/urfd/...'
# cfg['paths']['le2i_root'] = '/kaggle/input/le2i/...'
print('Kaggle:', on_kaggle())


Chạy cùng độ đo + cùng cửa sổ dung sai T cho M1/M2/M3 trên 4 protocol:
within_urfd, within_le2i, cross_A (train URFD→test Le2i), cross_B (ngược lại).


In [ ]:
!pip -q install ultralytics lap


In [ ]:
import torch, numpy as np
from src.detection_tracking import DetectorTracker
from src.data_splits import index_urfd, index_le2i
from src.pipeline import run_rule_on_testset, make_predict_fn, alarms_sequence_model, extract_id_crops
from src.datasets import make_gt_falls, video_fps
from src.evaluate import evaluate_events, aggregate_over_dataset, results_table
from src.models.model1_rule import RuleThresholds
dt = DetectorTracker(cfg['detection']['model'])
rows = []
# M1 trên test Le2i (within):
agg = run_rule_on_testset(index_le2i(cfg['paths']['le2i_root'])[:30], dt, RuleThresholds())
rows.append({'model':'M1','protocol':'within_le2i','precision':agg.precision,'recall':agg.recall,'f1':agg.f1,'false_alarm_per_hour':agg.false_alarm_per_hour,'mean_latency_sec':agg.mean_latency_sec})
# TODO: thêm M2/M3 (nạp checkpoint) + cross_A/B tương tự.
df = results_table(rows, 'outputs/results.csv'); print(df)


## Ablation cửa sổ T ∈ {0.5, 1.0, 2.0}s (nếu còn thời gian)
Lặp `evaluate_events(..., tolerance_sec=T)` và vẽ F1 theo T.
